# Gine & Karlan (Bulak) Replication — Group vs. Individual Liability Microfinance RCT

A partial, archive-constrained replication of Gine & Karlan's "Group versus Individual
Liability" field experiment (Bulak, Philippines), using the Harvard Dataverse release of the
underlying data.

This project is a deliberate structural contrast to a companion project analyzing a marketing
A/B test: treatment here was randomized at the **center** level (a center = several borrower
groups meeting together), not the individual level. That means valid inference requires
**cluster-robust standard errors** and **cluster-level permutation** — using individual rows as
independent observations would understate uncertainty and produce an invalid test.

**This repository is explicit, throughout, about a real data limitation:** the released archive
does not contain a direct key linking the client-level `grpname` field to the true center-level
randomization unit. The analysis below documents that limitation with an actual diagnostic check
(not an assumption), and uses a transparent, clearly-labeled fallback rather than silently
reporting results as if they were the paper's literal, correctly-specified replication.

## Setup

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import warnings

SEED = 20260901
N_PERM = 10_000

DATA_DIR = Path('../data')
OUT_DIR = Path('../outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = ['bulak_t1a_b.dta', 'bulak_figure3.dta', 'bulak_t1b.dta']
missing = [f for f in REQUIRED_FILES if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing required file(s) in {DATA_DIR.resolve()}: {', '.join(missing)}\n"
        f"See data/README.md for the Harvard Dataverse download link."
    )

In [11]:
def read_stata(path):
    """Read Stata files robustly, retaining numeric codes and handling legacy encodings."""
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UnicodeWarning)
        try:
            return pd.read_stata(path, convert_categoricals=False)
        except UnicodeDecodeError:
            return pd.read_stata(path, encoding='latin-1', convert_categoricals=False)

data = {}
for name in REQUIRED_FILES:
    df = read_stata(DATA_DIR / name)
    data[Path(name).stem] = df
    print(f'{name}: shape={df.shape}')

bulak_t1a_b.dta: shape=(26234, 73)
bulak_figure3.dta: shape=(1139, 93)
bulak_t1b.dta: shape=(43747, 206)


## Step 1 — The bridge problem

The true randomization unit is the **center**. To cluster correctly, `t1a_b.grpname` (a string,
client-level group name) needs to be linked to `figure3.center` / `figure3.cntid` (numeric center
and sub-unit identifiers). The cell below runs the actual diagnostic — it does not assume the link
exists or doesn't; it tests it directly.

In [12]:
fig = data['bulak_figure3'].copy()
loan = data['bulak_t1a_b'].copy()

pair = fig[['center', 'cntid']].drop_duplicates()
print(f'figure3 unique center-cntid pairs={len(pair)}; unique centers={pair.center.nunique()}; unique cntid={pair.cntid.nunique()}')

center_multiplicity = pair.groupby('center').cntid.nunique()
print('center->cntid multiplicity:', center_multiplicity.value_counts().sort_index().to_dict())

fig_key = fig.assign(center_key=fig['center'].astype(str).str.strip())[['center_key', 'cntid']].drop_duplicates()
loan_key = loan.assign(center_key=loan['grpname'].astype(str).str.strip())
literal = loan_key.merge(fig_key, on='center_key', how='left', indicator=True)
print('literal grpname=center merge:', literal['_merge'].value_counts(dropna=False).to_dict())

bridge_quality = pd.DataFrame({
    'metric': ['figure3_pair_rows', 'figure3_unique_centers', 'figure3_unique_cntid', 'literal_merge_rows', 'literal_merge_orphans'],
    'value': [len(pair), fig.center.nunique(), fig.cntid.nunique(), len(literal), int((literal._merge == 'left_only').sum())]
})
bridge_quality.to_csv(OUT_DIR / 'bridge_quality.csv', index=False)
bridge_quality

figure3 unique center-cntid pairs=163; unique centers=44; unique cntid=163
center->cntid multiplicity: {1: 16, 2: 2, 3: 1, 4: 8, 5: 3, 6: 5, 7: 9}
literal grpname=center merge: {'left_only': 26234, 'right_only': 0, 'both': 0}


,metric,value
0,figure3_pair_rows,163
1,figure3_unique_centers,44
2,figure3_unique_cntid,163
3,literal_merge_rows,26234
4,literal_merge_orphans,26234


**Result: every one of the 26,234 loan rows failed to match (`literal_merge_orphans = 26234`).**
`grpname` and `center` are not the same identifier system in this archive, under any string
cleaning. Separately, `center` alone maps to multiple `cntid` values (163 `cntid` across only 44
centers), so even a working bridge to `center` wouldn't uniquely identify the true randomization
unit. This is a documented property of the released archive, not a coding error — and it means
the true center-level cluster ID cannot be recovered from these files as supplied.

## Step 2 — Sample restriction and the fallback cluster definition

Following the paper's own sample restriction (pre-existing clients, first loan cycle after
assignment), then defining a fallback cluster using the only repeated key actually present in
`t1a_b`: `grpname`. Because `grpname` is *not* the true randomization unit, some `grpname` groups
contain a mix of treated and control clients — these are excluded from cluster-robust inference,
since both cluster-robust standard errors and cluster-level permutation require treatment to be
constant within a cluster.

In [13]:
d = loan.loc[(loan['old'] == 1) & (loan['cycle'] == 1)].copy()
d['treat'] = pd.to_numeric(d['treat'], errors='coerce')
d['dummypastdue30d'] = pd.to_numeric(d['dummypastdue30d'], errors='coerce')
d['prop_misswk_cycle_d'] = pd.to_numeric(d['prop_misswk_cycle_d'], errors='coerce')
d = d.loc[d.treat.isin([0, 1])].copy()

d['grpname_clean'] = d['grpname'].astype(str).str.strip()
d['cluster_treat_nunique'] = d.groupby('grpname_clean')['treat'].transform('nunique')
d['cluster_id'] = d['grpname_clean'].where(d['cluster_treat_nunique'] == 1)
d['cluster_assignment_valid'] = d['cluster_id'].notna()

d.to_csv(OUT_DIR / 'analysis_dataset.csv', index=False)
print('filtered rows:', len(d))
print('valid complete-treatment clusters:', d.loc[d.cluster_assignment_valid, 'cluster_id'].nunique())
print('rows with missing outcomes:', d[['dummypastdue30d', 'prop_misswk_cycle_d']].isna().sum().to_dict())

filtered rows: 4760
valid complete-treatment clusters: 631
rows with missing outcomes: {'dummypastdue30d': 0, 'prop_misswk_cycle_d': 0}


## Step 3 — Baseline balance check

Attempted, and honestly reported as not estimable: `t1b` (baseline survey) links to loan records
via `cid`, but `cid` is absent from `t1a_b`, so a baseline balance table cannot be constructed
from this archive without an additional crosswalk that isn't included in the release.

In [14]:
base = data['bulak_t1b'].copy()
balance_table = pd.DataFrame([{
    'variable': 'baseline_merge',
    'n_t1b_cid_nonmissing': int(base.cid.notna().sum()),
    'status': 'not estimable: cid absent from t1a_b'
}])
balance_table.to_csv(OUT_DIR / 'balance_table.csv', index=False)
balance_table

,variable,n_t1b_cid_nonmissing,status
0,baseline_merge,966,not estimable: cid absent from t1a_b


## Step 4 — Cluster-robust regression (point estimate + parametric SE)

Regression here is not used to get a *different* number than a plain difference-in-means — with a
single 0/1 predictor, OLS's fitted coefficient on `treat` is mathematically identical to the
group-mean difference. What regression provides is the ability to swap in a **clustered**
variance estimator (`cov_type='cluster'`), which allows for correlated errors within a
`grpname_clean` group rather than assuming every row is independent. Note this step deliberately
uses the *full* `grpname_clean` (not the treatment-pure `cluster_id`) — a clustered sandwich SE
doesn't require clusters to be treatment-pure the way the permutation test below does.

In [15]:
def ols_cluster(d, outcome):
    x = d[[outcome, 'treat', 'grpname_clean']].dropna().copy()
    model = sm.OLS(x[outcome], sm.add_constant(x['treat'])).fit(
        cov_type='cluster', cov_kwds={'groups': x['grpname_clean']}
    )
    return {
        'outcome': outcome, 'n': len(x), 'clusters': x.grpname_clean.nunique(),
        'control_mean': x.loc[x.treat == 0, outcome].mean(),
        'treatment_mean': x.loc[x.treat == 1, outcome].mean(),
        'ate': model.params['treat'], 'se_cluster': model.bse['treat'], 'p_cluster': model.pvalues['treat']
    }

## Step 5 — Cluster-level permutation test (non-parametric cross-check)

Uses only `cluster_assignment_valid` rows — i.e., only treatment-pure `grpname_clean` clusters —
since re-randomizing a mixed cluster has no honest label to assign it. Reassignment always draws
exactly `n_treat` clusters (the true observed count) on each iteration, which guarantees neither
side of a random draw is ever empty; this is the specific mechanism that avoids the
`ZeroDivisionError` that a naive per-row or unconstrained-per-cluster coin flip can produce.

In [16]:
def permutation_pvalue(d, outcome, n_perm=N_PERM, seed=SEED):
    x = d.loc[d.cluster_assignment_valid, [outcome, 'treat', 'cluster_id']].dropna().copy()
    observed = x.loc[x.treat == 1, outcome].mean() - x.loc[x.treat == 0, outcome].mean()

    cluster = x[['cluster_id', 'treat']].drop_duplicates().sort_values('cluster_id')
    labels = cluster.treat.to_numpy().astype(int)
    n_treat = int(labels.sum())

    rng = np.random.default_rng(seed)
    by_cluster = x.groupby('cluster_id')[outcome].agg(['sum', 'count'])
    vals = by_cluster.index.to_numpy()
    sums = by_cluster['sum'].to_numpy()
    counts = by_cluster['count'].to_numpy()

    stats = np.empty(n_perm)
    for i in range(n_perm):
        assign = np.zeros(len(vals), dtype=bool)
        assign[rng.choice(len(vals), size=n_treat, replace=False)] = True
        tm = sums[assign].sum() / counts[assign].sum()
        cm = sums[~assign].sum() / counts[~assign].sum()
        stats[i] = tm - cm

    p = (np.sum(np.abs(stats) >= abs(observed)) + 1) / (n_perm + 1)
    return observed, p

## Step 6 — Run both estimators on both outcomes, and save results

In [17]:
rows = []
for outcome in ['dummypastdue30d', 'prop_misswk_cycle_d']:
    r = ols_cluster(d, outcome)
    r['permutation_p'] = permutation_pvalue(d, outcome)[1]
    rows.append(r)

results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / 'results.csv', index=False)

sample_sizes = d.loc[d.cluster_assignment_valid].groupby('treat').agg(
    rows=('treat', 'size'), clusters=('cluster_id', 'nunique')
).reset_index()
sample_sizes.to_csv(OUT_DIR / 'sample_sizes.csv', index=False)

outcome_means = d.loc[d.cluster_assignment_valid].groupby('treat')[['dummypastdue30d', 'prop_misswk_cycle_d']].mean().reset_index()
outcome_means.to_csv(OUT_DIR / 'outcome_means.csv', index=False)

results

,outcome,n,clusters,control_mean,treatment_mean,ate,se_cluster,p_cluster,permutation_p
0,dummypastdue30d,4760,713,0.000429,0.000000,-0.000429,0.000426,0.313924,1.000000
1,prop_misswk_cycle_d,4760,713,0.046574,0.068961,0.022387,0.007229,0.001955,0.012499


## Step 7 — Interpretation

**`dummypastdue30d` (30-day delinquency):** treatment mean and control mean are both effectively
0 in this filtered sample, ATE is negligible (-0.0004), and both the cluster-robust p-value
(0.31) and permutation p-value (1.0) indicate no detectable effect. Read this as a floor/ceiling
effect in this particular filtered subsample rather than a meaningful null result — the outcome
barely varies at all within this slice of the data.

**`prop_misswk_cycle_d` (proportion of missed weekly payments):** control mean 0.0466, treatment
mean 0.0690 — individual-liability clients missed noticeably more payments than group-liability
clients in this fallback-cluster sample. Cluster-robust p-value ≈ 0.0020, permutation p-value ≈
0.0125. Both methods agree the difference is unlikely to be chance.

**The central caveat, restated plainly:** because the true center-level cluster identifier could
not be recovered from this archive (Step 1), the `prop_misswk_cycle_d` result above is a
same-direction diagnostic on a *degraded* cluster definition — not a literal reproduction of the
paper's own center-level estimate. It should be reported and used as such.